ETL - JUNÇÃO DAS TABELAS PARA ANÁLISE

In [1]:
import pandas as pd

clientes = pd.read_csv('Datasets/crm_clientes.csv',parse_dates=['data_cadastro'])
contatos = pd.read_csv('Datasets/crm_contatos.csv',parse_dates=['data_contato'])
propostas = pd.read_csv('Datasets/crm_propostas.csv',parse_dates=['data_envio'])

# display(clientes.sample(5))
# display(contatos.sample(5))
# display(propostas.sample(5))

contatos_agg = contatos.groupby(by='cliente_id').agg({'contato_id': 'count', 'tempo_resposta_horas': 'mean', 'data_contato': 'max'})
propostas_agg = propostas.groupby(by='cliente_id').agg({'proposta_id': 'count','valor_proposta': 'mean'})

clientes_contatos = pd.merge(left=clientes,right=contatos_agg,how='left',on='cliente_id',suffixes=('_clientes','_contatos'))

df_group = clientes_contatos.merge(right=propostas_agg,how='left',on='cliente_id',suffixes=('_clientes_contatos','propostas'))

df_group.sample(5)


target = (
    propostas
    .assign(comprou=lambda df: df['status_proposta'] == 'Ganha')
    .groupby('cliente_id')['comprou']
    .max()
    .reset_index()
)



df_group['valor_proposta'] = df_group['valor_proposta'].round(decimals=2)

df_group['tempo_resposta_horas'] = df_group['tempo_resposta_horas'].round(decimals=0)

df_group = df_group.fillna(0)

df_group = df_group.merge(right=target,how='left', on='cliente_id')

df_group = df_group.fillna(False)

df_group = df_group[['cliente_id','tipo_cliente','proposta_id','contato_id','tempo_resposta_horas','valor_proposta','data_contato','comprou']]


df_group.columns = ['cliente_id','tipo_cliente','qtde_proposta','qtde_contato', 'media_tempo_proposta','media_valor_proposta','ultimo_contato','comprou']

x = df_group[['cliente_id','qtde_proposta','qtde_contato', 'media_tempo_proposta','media_valor_proposta']]

y = df_group['comprou']

# x = x.fillna(0)



C:\Users\Igor\AppData\Local\Temp\ipykernel_26084\643576296.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_group = df_group.fillna(False)


SERAPAÇÃO DO DATASET DE TREINAMENTO E DATASET DE TESTE

In [2]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=16)




MODELO DE REGRESSÃO LOGISTICA

In [3]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(random_state=16)

# fit
logreg.fit(x_train, y_train)

y_pred = logreg.predict(x_test)
y_pred_proba = logreg.predict_proba(x_test)

# display(y_pred_proba)



teste = pd.DataFrame(x_test).reset_index().drop('index',axis=1)

probs = pd.DataFrame(y_pred_proba)

# display(probs)



final = pd.concat([teste,probs],axis = 1)

final.rename(columns={0:'%_não_comprar', 1: '%_comprar'},inplace=True)



final['%_não_comprar'] = (final['%_não_comprar']*100).round(decimals=2)
final['%_comprar'] = (final['%_comprar']*100).round(decimals=2)

final = final.sort_values(by='%_comprar',ascending=False)

final = final[final['%_comprar']>=40].head(10)

final



,cliente_id,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar
98,289,9.0,6.0,47.0,22015.89,0.77,99.23
28,268,6.0,7.0,46.0,24338.00,5.53,94.47
57,190,6.0,7.0,37.0,22786.00,5.53,94.47
45,223,5.0,8.0,50.0,24869.20,9.89,90.11
50,106,5.0,3.0,68.0,25220.60,10.34,89.66
31,256,5.0,7.0,54.0,20947.80,10.46,89.54
66,290,5.0,4.0,59.0,23875.80,11.09,88.91
33,292,5.0,4.0,68.0,18629.80,11.28,88.72
75,417,5.0,4.0,58.0,27571.60,11.35,88.65
21,133,4.0,8.0,46.0,29514.75,17.03,82.97


Verificar se o que o modelo disse está coerente.

In [24]:

y_proba = logreg.predict_proba(x_test)[:, 1]


from sklearn.metrics import confusion_matrix

y_pred_custom = (y_proba >= 0.38).astype(int)
matrix = pd.DataFrame(confusion_matrix(y_test, y_pred_custom))

matrix.rename(columns={0:'Modelo_Disse_NaoComprou',1:'Modelo_Disse_Comprou'},inplace=True)
matrix.rename({0:'Nao_Comprou',1:'Comprou'},inplace=True)
matrix



,Modelo_Disse_NaoComprou,Modelo_Disse_Comprou
Nao_Comprou,21,23
Comprou,5,51


In [236]:
# clientes com valor medio da proposta maior e alta % de comprar:
display(final.sort_values(by=['media_valor_proposta','%_comprar'],ascending=[False,False]).head(10))

# clientes com alta % de comprar:
display(final.sort_values(by='%_comprar',ascending=False).head(10))

,cliente_id,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar
21,133,4.0,8.0,46.0,29514.75,17.03,82.97
75,417,5.0,4.0,58.0,27571.60,11.35,88.65
50,106,5.0,3.0,68.0,25220.60,10.34,89.66
45,223,5.0,8.0,50.0,24869.20,9.89,90.11
28,268,6.0,7.0,46.0,24338.00,5.53,94.47
66,290,5.0,4.0,59.0,23875.80,11.09,88.91
57,190,6.0,7.0,37.0,22786.00,5.53,94.47
98,289,9.0,6.0,47.0,22015.89,0.77,99.23
31,256,5.0,7.0,54.0,20947.80,10.46,89.54
33,292,5.0,4.0,68.0,18629.80,11.28,88.72


,cliente_id,qtde_proposta,qtde_contato,media_tempo_proposta,media_valor_proposta,%_não_comprar,%_comprar
98,289,9.0,6.0,47.0,22015.89,0.77,99.23
28,268,6.0,7.0,46.0,24338.00,5.53,94.47
57,190,6.0,7.0,37.0,22786.00,5.53,94.47
45,223,5.0,8.0,50.0,24869.20,9.89,90.11
50,106,5.0,3.0,68.0,25220.60,10.34,89.66
31,256,5.0,7.0,54.0,20947.80,10.46,89.54
66,290,5.0,4.0,59.0,23875.80,11.09,88.91
33,292,5.0,4.0,68.0,18629.80,11.28,88.72
75,417,5.0,4.0,58.0,27571.60,11.35,88.65
21,133,4.0,8.0,46.0,29514.75,17.03,82.97


Teste de modelo para ver acurácia: GradientBoostingClassifier

In [ ]:

from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gb.fit(x_train, y_train)

gb_proba = gb.predict_proba(x_test)[:, 1]




0.6948051948051948
0.7784090909090909


Teste de modelo para ver acurácia: RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

rf.fit(x_train, y_train)

rf_proba = rf.predict_proba(x_test)[:, 1]



0.6785714285714286


Verificar qual modelo melhor se alinhou ao dataset

In [28]:
from sklearn.metrics import roc_auc_score

print("gradient boosting:", roc_auc_score(y_test, gb_proba))
print("logistic regression:", roc_auc_score(y_test, y_proba))
print("random forest:", roc_auc_score(y_test, rf_proba))

scores = [("gradient boosting", roc_auc_score(y_test, gb_proba)), ("logistic regression", roc_auc_score(y_test, y_proba)), ("random forest", roc_auc_score(y_test, rf_proba))]

melhor_modelo = sorted(scores, key=lambda x: x[1], reverse=True)[0]
print(f'O modelo que melhor se alinhou ao dataset foi o modelo de {melhor_modelo[0]}: {melhor_modelo[1]}')

gradient boosting: 0.6948051948051948
logistic regression: 0.7784090909090909
random forest: 0.6785714285714286
O modelo que melhor se alinhou ao dataset foi o modelo de logistic regression: 0.7784090909090909
